# 021 — Disaggregation post-processing (IML-based)

Turns the **IML-based** seismic-hazard disaggregations run by
`020-disaggregation.ipynb` into flat, provenance-tracked pickles for record
selection.

For each `(IM definition, epsilon truncation)` pair the notebook:

1. Resolves the OpenQuake `calc_id` of every target-IML disaggregation from
   `wp1/disagg_manifest.json` — no hardcoded integers.
2. Collates them into a flat dictionary keyed by site index,
   `disagg_data[site][imt][iml] -> DataFrame`, and writes it **one pickle per
   site** (`site_NNN.pickle` + an `_index.json` of the native IML keys) rather
   than one multi-GB blob — see `scripts/disagg_shards.py`.
3. Builds a flat disagg-**stats** DataFrame, one row per `(site, imt, iml)`.
4. Writes both through the project's provenance cache (`cache_utils` /
   `disagg_shards`), so a re-run reloads unchanged artifacts and refuses stale ones.

Sharding by site is what lets the downstream record selection (nbs `031`–`033`)
fingerprint each `(site, iml)` stripe against **its own site's shard**: re-running the
disaggregation for one site then leaves every other site's already-selected stripes
valid, instead of invalidating all ~450 of them.

The **occurrence** disaggregation `P(m|X=x)` needs the hazard-curve slope at the
target IML. An `iml_disagg` datastore holds only a single intensity level, so the
slope is taken from the full-resolution hazard-curve pickles produced by
`004-psha_results.ipynb` — after asserting they share the disaggregation runs'
source model / logic tree (`oqhelpers.assert_shared_provenance`).

## Prerequisites

- **`020-disaggregation.ipynb` must have run** (with `DRY_RUN = False`), so every
  `AvgSA_{im}_disagg_eps{eps}_{iml}` entry (e.g. `AvgSA_03_disagg_eps4_0pt285`)
  exists in `hazard_models/eshm20/wp1/disagg_manifest.json` and its datastore
  (`calc_<id>.hdf5`) is present in the local `oqdata` directory. The datastores
  are machine-local and rebuildable; the manifest is the git-tracked pointer.
- **`004-psha_results.ipynb` must have run** (with `SAVE = True`), producing the
  hazard-curve pickles `AvgSA_{im}_hazard_curves_60sites_{3,4,5}sig.pickle` that
  supply the occurrence slope. Their `eps{N}` ↔ `{N}sig` PSHA calculations
  (`wp1/psha_manifest.json`) must share the disaggregation source model — this is
  asserted at run time.
- Target IMLs come from `data_processed/03_site_hazard/AvgSA_{03,06}_imls_for_disaggregation.csv`
  (written by `017-disagg_imls_for_msa_stripes.ipynb`).

## Dependencies

**Upstream:** `017` (target IMLs) → `020` (disagg runs) and `003`→`004` (PSHA
hazard curves). **Downstream:** the record-selection setup
(`setup_AvgSA0{3,6}_gm_selection.py`) and notebooks `031`–`036` consume the
pickles written here.

The notebook is **data-driven from the manifest**: it processes whatever
`(IM, eps)` pairs are present, so `AvgSA_06` is picked up automatically once its
disaggregations are run (its IML file does not exist yet).

## Output structure

Written to `data_processed/03_site_hazard/`, one `(data, stats, excluded)` set per
`(IM, eps)`:

| File | Object |
|---|---|
| `AvgSA_{im}_disagg_data_wp1sites_eps{eps}/site_NNN.pickle` | `dict[imt][iml] -> DataFrame` for one site |
| `AvgSA_{im}_disagg_data_wp1sites_eps{eps}/_index.json` | `{site: {imt: [iml, ...]}}` — the native IML keys, readable without opening a shard |
| `AvgSA_{im}_disagg_stats_wp1sites_eps{eps}.pickle` | flat `DataFrame`, one row per `(site, imt, imtl)` |
| `AvgSA_{im}_excluded_imls_wp1sites_eps{eps}.json` | `{site_id: [imls]}` — zero-hazard IMLs dropped from stats |

- `iml` keys are the target level in g at 6 significant figures (`float`).
- Each disaggregation `DataFrame` has columns `TRT, Mag, Dist, Eps, Z,
  P(X>x|T,m), nu_m, P(m|X>x), P(m|X=x)`.
- Each stats row has `site_id, lat, lon, seismicity, region, imt, imtl, poe,
  mafe, rtp` — where `poe` / `mafe` / `rtp` are the annual PoE, mean annual
  frequency of exceedance and return period at `imtl`, interpolated from the
  site's matching-eps hazard curve — plus per-TRT `"<TRT> [%]"` proportions and
  `Mag_mean` / `Dist_mean`.
- IMLs with zero hazard (`mafe <= 0`, beyond the epsilon-truncated maximum ground
  motion) are **excluded from the stats** but **kept in the data**, and listed in
  the `..._excluded_imls_...json` file (see below).
- A `<file>.manifest.json` provenance sidecar is written next to each pickle, including
  every per-site shard (all shards of one build share the same fingerprint).

In [1]:
%load_ext autoreload
%autoreload 2

## 0. Setup

In [ ]:
import json
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
from openquake.commonlib.datastore import read

from phd_project.config import config
from phd_project.scripts import oq_runner, oqhelpers
from phd_project.scripts import cache_utils, disagg_shards

cfg = config.load_config()

In [3]:
# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------
HAZ_DIR = cfg["proc_data"]["site_hazard"]
WP1_DIR = cfg["hazard_models"]["eshm20_wp1"]
DISAGG_MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_disagg_manifest"]
PSHA_MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_psha_manifest"]
SITES_FP = cfg["results"]["selected_sites_csv"]

# IM definition -> target-IML file (as written by 017).
IML_FILES = {
    "03": cfg["proc_data"]["disagg_imls_AvgSA_03"],
    # "06": cfg["proc_data"]["disagg_imls_AvgSA_06"],   # discontinued
}

# eps truncation level -> hazard-curve pickle suffix (004 writes _{N}sig).
TRUNC_SUFFIX = {4: "4sig"}

DISAGG_TYPE = "TRT_Mag_Dist_Eps"
TRADITIONAL = True     # P(m|X>x)
OCCURENCE = True       # P(m|X=x), needs the hazard-curve slope

# FORCE_RECOMPUTE: rebuild every pickle even when its inputs are unchanged.
FORCE_RECOMPUTE = False

print(f"hazard dir:      {HAZ_DIR}")
print(f"disagg manifest: {DISAGG_MANIFEST_FP}")
print(f"psha manifest:   {PSHA_MANIFEST_FP}")
print(f"TRADITIONAL={TRADITIONAL}, OCCURENCE={OCCURENCE}, "
      f"FORCE_RECOMPUTE={FORCE_RECOMPUTE}")

hazard dir:      C:\Users\clemettn\Documents\phd\data_processed\03_site_hazard
disagg manifest: C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1\disagg_manifest.json
psha manifest:   C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1\psha_manifest.json
TRADITIONAL=True, OCCURENCE=True, FORCE_RECOMPUTE=False


## 1. Load inputs

The manifests, the `{name: calc_id}` map, the target IMLs, and the site metadata
(`results/01_site_selection/sites.csv`, whose row order matches the datastore
site order 0..N-1). The `(IM, eps)` pairs to process are discovered from the
manifest entry names.

In [4]:
disagg_manifest = oq_runner.load_manifest(DISAGG_MANIFEST_FP)
psha_manifest = oq_runner.load_manifest(PSHA_MANIFEST_FP)
calc_ids = oq_runner.load_calc_ids(DISAGG_MANIFEST_FP)

# Site metadata: sites.csv already carries lat/lon/seismicity/region and is in the
# same row order (0..N-1) as the disaggregation datastores.
site_metadata = pd.read_csv(SITES_FP)
n_sites = len(site_metadata)

# Target IMLs per IM definition (skip an IM whose file is not written yet).
imls_by_im = {}
for im, fp in IML_FILES.items():
    fp = Path(fp)
    if fp.is_file():
        imls_by_im[im] = oq_runner.load_imls(fp)
    else:
        print(f"[skip] AvgSA {im}: no IML file at {fp.name}")

# Discover the (im, eps) pairs present, each an ordered list of (iml_token, name).
# The IML token (e.g. '0pt285') is matched back to the full-precision IML file below.
pair_re = re.compile(r"^AvgSA_(?P<im>\d+)_disagg_eps(?P<eps>\d+)_(?P<tok>\d+pt\d+)$")
pairs = {}
for name in calc_ids:
    m = pair_re.match(name)
    if m:
        pairs.setdefault((m["im"], int(m["eps"])), []).append((m["tok"], name))
for key in pairs:
    pairs[key].sort(key=lambda t: float(t[0].replace("pt", ".")))

print(f"{n_sites} sites; {len(pairs)} (IM, eps) pairs present:")
for (im, eps), items in sorted(pairs.items()):
    lo, hi = calc_ids[items[0][1]], calc_ids[items[-1][1]]
    print(f"  AvgSA_{im} eps{eps}: {len(items)} imls (calc {lo}..{hi})")

60 sites; 1 (IM, eps) pairs present:
  AvgSA_03 eps4: 23 imls (calc 102..111)


## 2. Collate and cache the disaggregation data and stats

For each `(IM, eps)` pair: assert the hazard curves share the disaggregation
source model, then collate the per-IML disaggregations into the flat dict and
build the stats DataFrame — the stats through `cache_utils.load_or_compute`, the data
through `disagg_shards.load_or_compute_shards` (same branches, one pickle per site).

The hazard curves are always needed here: the stats `poe`/`mafe`/`rtp` are read
from them at each IML, and (when `OCCURENCE`) they also supply the occurrence
slope. The provenance fingerprint therefore covers the per-IML disagg configs,
the site model, the target-IML file, the calc ids, and the hazard-curve pickle. A
changed input raises `StaleCacheError`; set `FORCE_RECOMPUTE = True` to overwrite.

### Excluded zero-hazard IMLs

The MSA target IMLs (from `017`) go up to the highest fragility point, and at the
tighter epsilon truncations the strongest stripes can exceed the **maximum ground
motion the GMM can produce**. Above that cap the mean hazard curve gives
`MAFE = 0` (annual rate zero → `poe = 0`, `rtp = ∞`), so the intensity level is
not a physically meaningful hazard scenario and its disaggregation cannot be
summarised as a return period.

For every `(site, imt, iml)` with `mafe <= 0`, `oqhelpers.get_iml_disagg_stats`
**omits the stats row** (so `disagg_stats` holds only real hazard levels), while
the disaggregation itself is **kept in `disagg_data`**. The dropped IMLs are
recorded per pair as `{site_id: [imls]}` in
`AvgSA_{im}_excluded_imls_wp1sites_eps{eps}.json` and printed below.

> Note: `mafe <= 0` (zero hazard) is broader than "the OpenQuake disaggregation
> array is all-zero". At most of these IMLs the disagg run still returns a
> negligible non-zero rate (~1e-9), but the hazard curve's truncated tail floors
> to zero — either way the return period is effectively infinite, so the level is
> excluded.

In [ ]:
results = {}   # (im, eps) -> {"data": ..., "stats": ..., "excluded": ...}

for (im, eps), items in sorted(pairs.items()):
    if im not in imls_by_im:
        raise FileNotFoundError(
            f"AvgSA_{im} has disagg calcs but no IML file {IML_FILES[im]}")

    names = [name for _, name in items]
    pair_calc_ids = [calc_ids[name] for name in names]
    # Recover the full-precision IML for each name by matching its token back to
    # the target-IML file (the name only carries the 3 dp token).
    tok_to_iml = {oq_runner._iml_fname_token(v): float(v) for v in imls_by_im[im]}
    imls = [tok_to_iml[tok] for tok, _ in items]
    assert len(imls) == len(pair_calc_ids)

    # --- hazard curves (matching eps): supply the stats poe/mafe/rtp at each IML
    # and, when OCCURENCE, the occurrence slope. Assert they share the disagg
    # runs' source model before use.
    oqhelpers.assert_shared_provenance(disagg_manifest, psha_manifest, im, eps)
    hc_fp = HAZ_DIR / f"AvgSA_{im}_hazard_curves_60sites_{TRUNC_SUFFIX[eps]}.pickle"
    with open(hc_fp, "rb") as f:
        hcurves = pickle.load(f)

    # The disagg data is stored PER SITE (site_NNN.pickle + _index.json in this
    # directory), not as one multi-GB pickle -- see scripts/disagg_shards.py. The
    # stats stay a single small pickle.
    shard_dir = HAZ_DIR / f"AvgSA_{im}_disagg_data_wp1sites_eps{eps}"
    stats_fp = HAZ_DIR / f"AvgSA_{im}_disagg_stats_wp1sites_eps{eps}.pickle"
    excl_fp = HAZ_DIR / f"AvgSA_{im}_excluded_imls_wp1sites_eps{eps}.json"

    # --- provenance fingerprint (shared by the data and stats artifacts)
    fp_inputs = {f"config_{tok}": WP1_DIR / oq_runner.disagg_config_name(im, eps, iml)
                 for (tok, _), iml in zip(items, imls)}
    fp_inputs["sites_csv"] = SITES_FP
    fp_inputs["iml_csv"] = IML_FILES[im]
    fp_inputs["calc_ids"] = str(pair_calc_ids)
    fp_inputs["hazard_curves"] = hc_fp
    fp_dict = cache_utils.fingerprint(**fp_inputs)

    # Every shard of one build shares fp_dict: they come from one disagg run. The
    # collation still builds all sites in memory (it reads one datastore per IML,
    # each carrying every site) -- sharding pays off on the read side downstream.
    disagg_data = disagg_shards.load_or_compute_shards(
        shard_dir, fp_dict,
        lambda: oqhelpers.collate_disagg_by_iml(
            pair_calc_ids, imls, disagg_type=DISAGG_TYPE,
            traditional=TRADITIONAL, occurence=OCCURENCE,
            hcurves=hcurves, reader=read),
        force_recompute=FORCE_RECOMPUTE)

    disagg_stats = cache_utils.load_or_compute(
        stats_fp, fp_dict,
        lambda: oqhelpers.get_iml_disagg_stats(disagg_data, site_metadata, hcurves),
        force_recompute=FORCE_RECOMPUTE)

    # --- zero-hazard (mafe <= 0) IMLs: dropped from stats, kept in data, recorded
    excluded = oqhelpers.get_excluded_imls(disagg_data, hcurves)
    with open(excl_fp, "w") as f:
        json.dump({str(s): imls for s, imls in excluded.items()}, f, indent=2)

    results[(im, eps)] = {"data": disagg_data, "stats": disagg_stats,
                          "excluded": excluded}
    n_excl = sum(len(v) for v in excluded.values())
    print(f"AvgSA_{im} eps{eps}: {shard_dir.name}/ ({len(disagg_data)} site shards) "
          f"+ {stats_fp.name} "
          f"({len(disagg_stats)} stat rows; {n_excl} zero-hazard imls excluded "
          f"across {len(excluded)} sites -> {excl_fp.name})")
    for s, imls in sorted(excluded.items()):
        print(f"    site {s}: {imls}")

## 3. Sanity checks

Confirm the flat structure, the per-IML keys, the DataFrame columns, and that the
TRT proportions sum to ~100 % per stats row.

In [6]:
(im, eps) = sorted(results)[0]
data = results[(im, eps)]["data"]
stats = results[(im, eps)]["stats"]

site0 = sorted(data)[0]
imt0 = list(data[site0])[0]
imls_present = sorted(data[site0][imt0])
print(f"AvgSA_{im} eps{eps}: {len(data)} sites keyed 0..{max(data)}")
print(f"  site {site0}, imt {imt0!r}: {len(imls_present)} imls -> {imls_present}")

df0 = data[site0][imt0][imls_present[0]]
print("  disagg df columns:", list(df0.columns))
print("  stats columns:    ", list(stats.columns))

pct_cols = [c for c in stats.columns if c.endswith("[%]")]
print("  TRT % row-sum summary:")
print(stats[pct_cols].sum(axis=1).describe()[["min", "mean", "max"]].to_string())

AvgSA_03 eps4: 60 sites keyed 0..59
  site 0, imt 'AvgSA': 23 imls -> [0.1645, 0.2109, 0.2423, 0.27, 0.285, 0.31, 0.33, 0.36, 0.4, 0.45, 0.4898, 0.5, 0.55, 0.6049, 0.65, 0.7139, 0.7892, 0.8, 0.8904, 0.95, 1.0112, 1.15, 1.3368]
  disagg df columns: ['TRT', 'Mag', 'Dist', 'Eps', 'Z', 'P(X>x|T,m)', 'nu_m', 'P(m|X>x)', 'P(m|X=x)']
  stats columns:     ['site_id', 'lat', 'lon', 'seismicity', 'region', 'imt', 'imtl', 'poe', 'mafe', 'rtp', 'Craton [%]', 'Non-Subduction Deep [%]', 'Shallow Default [%]', 'Subduction Inslab [%]', 'Subduction Interface [%]', 'Volcanic [%]', 'Mag_mean', 'Dist_mean']
  TRT % row-sum summary:
min      99.990000
mean     99.999985
max     100.010000
